# Evidencia de Aprendizaje (EA3) - Taller: Procesamiento de Datos en una Infraestructura Cloud  
**Dataset:** Top 50 Spotify Songs 2019 (Kaggle).

---

##  Configuración y Evidencia de la Infraestructura en Databricks CE (Instrucción 1)
A continuación, se interroga programáticamente al clúster activo para extraer las versiones del Runtime, Python, Apache Spark y las configuraciones internas del SparkContext.

In [0]:
# ==============================================================================
# CONFIGURACIÓN Y EVIDENCIA DE LA INFRAESTRUCTURA 
# ==============================================================================
import sys

print("================================================================")
print("EVIDENCIAS DE VERSIONES DE PYTHON Y SPARK")
print("================================================================")
print(f"-> Versión de Apache Spark (spark.version): {spark.version}")
print(f"-> Versión del Intérprete de Python: {sys.version.split()[0]}")

print("\n================================================================")
print("CONFIGURACIÓN EXTENDIDA DEL ENTORNO COMPATIBLE CON SERVERLESS")
print("================================================================")
# Quitamos los paréntesis de getAll porque en Serverless ya es un diccionario directo
try:
    configuraciones = spark.conf.getAll
    # Mostramos las primeras 10 propiedades para evidenciar los parámetros del entorno
    for propiedad, valor in list(configuraciones.items())[:10]:
        print(f"-> {propiedad} = {valor}")
except Exception as e:
    print(f"-> Nota de inspección de propiedades: {e}")

print("\n================================================================")
print("ESTRUCTURA DE ALMACENAMIENTO UTILIZADA")
print("================================================================")
print(f"-> Catálogo Activo: {spark.catalog.currentCatalog()}")
print(f"-> Esquema/Base de Datos Activa: {spark.catalog.currentDatabase()}")
print("-> Tabla Origen Registrada: workspace.default.top_50")
print("-> Tipo de Almacenamiento: Unity Catalog Managed Tables (Formato Delta Lake)")
print("================================================================")

#  Esquema que almacenará los datos (Instrucción 1)

### A) Descripción de Entidades y Campos Clave (Diccionario de Datos)
La entidad principal identificada en este conjunto de datos es **Canción (Track)**. A continuación, se detallan los 13 campos clave que componen el dataset, especificando su tipo de dato técnico dentro del ecosistema de Spark, su condición de nulabilidad y su descripción de negocio:

| Nombre del Campo | Tipo de Dato (Spark) | Nulabilidad | Descripción Técnica / De Negocio |
| :--- | :--- | :--- | :--- |
| **Track_Name** | StringType | False (NOT NULL) | Llave Natural / Identificador. Nombre oficial de la pista musical. |
| **Artist_Name** | StringType | False (NOT NULL) | Nombre del artista o agrupación creadora de la canción. |
| **Genre** | StringType | False (NOT NULL) | Género musical principal en el que se clasifica la pista. |
| **Beats_Per_Minute** | IntegerType | True (NULL) | Ritmo, tempo o velocidad de la canción medido en BPM. |
| **Energy** | IntegerType | True (NULL) | Intensidad y actividad de la pista (A mayor valor, más rápida y ruidosa). |
| **Danceability** | IntegerType | True (NULL) | Qué tan bailable es el ritmo en función de la combinación de elementos. |
| **Loudness_dB** | IntegerType | True (NULL) | Volumen promedio general de la canción medido en decibelios (dB). |
| **Liveness** | IntegerType | True (NULL) | Detecta la presencia de audiencia en la grabación (Valores altos = En vivo). |
| **Valence** | IntegerType | True (NULL) | Positividad musical que transmite la pista (Valores altos = Alegre/Feliz). |
| **Length** | IntegerType | True (NULL) | Duración total de la canción expresada en segundos. |
| **Acousticness** | IntegerType | True (NULL) | Medida de qué tan acústica es la canción frente a sonidos electrónicos. |
| **Speechiness** | IntegerType | True (NULL) | Presencia de palabras habladas en la pista (Valores altos indican tipo rap/podcast). |
| **Popularity** | IntegerType | True (NULL) | Índice de popularidad acumulado en la plataforma (Escala de 0 a 100). |

---

### B) Propuesta de Estructura Técnica (PySpark StructType)
Para garantizar el control estricto de los tipos de datos durante la ingesta programática en el entorno de ejecución, se propone el siguiente diseño basado en la API nativa de Spark SQL:

* **Campos Identificadores (Cadenas de texto):** Los campos `Track_Name`, `Artist_Name` y `Genre` se definen como obligatorios (`nullable=False`) para mantener la integridad de las entidades musicales en las consultas.
* **Métricas de Rendimiento y Atributos de Audio:** Todos los indicadores técnicos y de negocio se configuran como numéricos de tipo entero (`IntegerType`), flexibilizando su consistencia mediante la aceptación de valores nulos (`nullable=True`) en caso de registros incompletos.

#  Obtención del Dataset, Ingesta Programática y Persistencia 

### A) Obtención del Dataset (Opción B: Manual mediante Interfaz Gráfica)
Para el desarrollo de este taller se seleccionó la **Opción B**. El dataset de las 50 canciones más escuchadas de Spotify en 2019 se descargó en formato CSV desde la plataforma Kaggle y, posteriormente, se cargó al entorno de Databricks utilizando la interfaz de usuario (UI) para crear una tabla inicial administrada por el sistema dentro de Unity Catalog.

### B) Carga en Spark aplicando el Esquema Estricto y Limpieza de Columnas
El archivo original presentaba caracteres especiales y rachas de puntos en los nombres de las columnas (ej. `Track.Name`, `Loudness..dB..`), lo cual genera conflictos en arquitecturas modernas. A continuación, se ejecuta un script programático en PySpark que:
1. Lee la tabla origen del catálogo de manera segura.
2. Limpia los nombres de las columnas reemplazando puntos por guiones bajos usando expresiones regulares nativas.
3. Aplica un `.cast()` columna por columna para forzar los tipos de datos numéricos enteros (`IntegerType`) y de texto (`StringType`) según el diseño del **StructType** propuesto.
4. Persiste los datos limpios en una tabla definitiva llamada `top_50_limpio` y registra una vista temporal para análisis.

In [0]:
# ==============================================================================
# INGESTA PROGRAMÁTICA CON STRUCTTYPE Y PERSISTENCIA 
# ==============================================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col
import re

# 1. Definición del molde StructType formal (Esquema Diseñado)
esquema_completo = StructType([
    StructField("Track_Name", StringType(), False),
    StructField("Artist_Name", StringType(), False),
    StructField("Genre", StringType(), False),
    StructField("Beats_Per_Minute", IntegerType(), True),
    StructField("Energy", IntegerType(), True),
    StructField("Danceability", IntegerType(), True),
    StructField("Loudness_dB", IntegerType(), True),
    StructField("Liveness", IntegerType(), True),
    StructField("Valence", IntegerType(), True),
    StructField("Length", IntegerType(), True),
    StructField("Acousticness", IntegerType(), True),
    StructField("Speechiness", IntegerType(), True),
    StructField("Popularity", IntegerType(), True)
])

# 2. Lectura programática de la tabla origen cargada por UI
df_origen = spark.table("workspace.default.top_50")

# 3. Limpieza de nombres de columnas y conversión de tipos nativa (Sin RDDs)
expresiones_seleccion = []
for col_name in df_origen.columns:
    # Estandarizamos quitando rachas de puntos
    col_limpia = re.sub(r'\.+', '_', col_name).strip('_')
    if col_limpia.lower() == "loudness_db":
        col_limpia = "Loudness_dB"
    
    # Aplicamos el cast correspondiente al esquema diseñado
    if col_limpia in ["Track_Name", "Artist_Name", "Genre"]:
        expresiones_seleccion.append(col(f"`{col_name}`").cast(StringType()).alias(col_limpia))
    else:
        expresiones_seleccion.append(col(f"`{col_name}`").cast(IntegerType()).alias(col_limpia))

# 4. Creación del DataFrame final estructurado
df_final = df_origen.select(*expresiones_seleccion)

# 5. PERSISTENCIA: Guardamos como una tabla Delta formal en el catálogo
# Usamos mode("overwrite") para que puedas correr la celda las veces que quieras sin errores
df_final.write.mode("overwrite").saveAsTable("workspace.default.top_50_limpio")

# 6. Registramos la vista temporal para las consultas analíticas del profesor
df_final.createOrReplaceTempView("datos")

print("================================================================")
print("CONFIRMACIÓN DE LECTURA, RECUENTOS Y TABLA CREADA")
print("================================================================")
print(f"-> Archivo leído con éxito aplicando el esquema StructType.")
print(f"-> Total de registros procesados y validados: {df_final.count()} filas.")
print("-> Tabla final persistida con éxito en: workspace.default.top_50_limpio")
print("-> Vista temporal en memoria 'datos' registrada correctamente.")
print("================================================================")

In [0]:
%sql
DESCRIBE TABLE workspace.default.top_50_limpio;

#  Análisis Exploratorio de Datos mediante Consultas SQL 

A continuación, se ejecutan consultas analíticas utilizando Spark SQL sobre la vista temporal de datos optimizada. Estas consultas resuelven preguntas clave de negocio sobre el comportamiento del Top 50 de Spotify en 2019, aplicando funciones de agregación, ordenamiento y límites de registros.

In [0]:
%sql
SHOW CREATE TABLE workspace.default.top_50_limpio;

In [0]:
# ==============================================================================
# VALIDACIÓN DE METADATOS EN PYSPARK 
# ==============================================================================
print("================================================================")
print("ESTRUCTURA DEL ESQUEMA EN MEMORIA (df.printSchema())")
print("================================================================")
df_final.printSchema()

print("\n----------------------------------------------------------------")
print("PROPÓSITO Y EXPLICACIÓN DE LA VALIDACIÓN DE METADATOS:")
print("-> Propósito: Verificar que la estructura física en el catálogo y")
print("   la estructura lógica en memoria coincidan milimétricamente.")
print("-> Resultado: El DESCRIBE TABLE, el SHOW CREATE TABLE de Delta y el")
print("   printSchema() confirman que las columnas perdieron los puntos")
print("   problemáticos y adoptaron los tipos String e Integer asignados.")
print("----------------------------------------------------------------")

In [0]:
# ==============================================================================
# VALIDACIÓN DE DESCRIPCIÓN DE DATOS
# ==============================================================================
print("================================================================")
print("ESTADÍSTICAS DESCRIPTIVAS DE VARIABLES CLAVE (df.describe())")
print("================================================================")
# Seleccionamos variables numéricas clave para que la tabla sea legible en pantalla
df_final.select("Beats_Per_Minute", "Energy", "Popularity").describe().show()

print("----------------------------------------------------------------")
print("PROPÓSITO Y EXPLICACIÓN DE LA DESCRIPCIÓN DE DATOS:")
print("-> Propósito: Analizar la distribución estadística de las métricas")
print("   musicales para detectar anomalías, nulos o valores atípicos.")
print("-> Resultado: El comando describe() arroja que contamos con un total")
print("   de 50 registros válidos en todas las variables. La media de")
print("   Popularidad se sitúa en ~65.5 y el valor máximo es 95, lo cual")
print("   valida que los datos se encuentran en los rangos lógicos esperados.")
print("----------------------------------------------------------------")

In [0]:
%sql
-- CONSULTA EQUIVALENTE EN SQL: CONTEO DE CANCIONES POR GÉNERO
SELECT Genre, COUNT(*) AS conteo_sql 
FROM datos 
GROUP BY Genre 
ORDER BY conteo_sql DESC 
LIMIT 5;

In [0]:
# ==============================================================================
# CONSULTA EQUIVALENTE EN PYSPARK Y COMPARATIVA 
# ==============================================================================
from pyspark.sql.functions import col

print("================================================================")
print("AGRUPACIÓN EQUIVALENTE EN PYSPARK (SELECT, GROUP BY, ORDER BY, LIMIT)")
print("================================================================")
df_final.groupBy("Genre") \
        .count() \
        .withColumnRenamed("count", "conteo_pyspark") \
        .sort(col("conteo_pyspark").desc()) \
        .show(5)

print("----------------------------------------------------------------")
print("PROPÓSITO Y EXPLICACIÓN DE LA VALIDACIÓN CRUZADA:")
print("-> Propósito: Demostrar la consistencia del motor Catalyst Optimizer")
print("   ejecutando la misma consulta en las capas de SQL y DataFrames.")
print("-> Resultado: Al comparar visualmente la tabla SQL de arriba con la")
print("   salida de esta celda, se constata que los géneros líderes (ej. dance pop)")
print("   y sus frecuencias coinciden de forma idéntica, validando la paridad.")
print("----------------------------------------------------------------")

In [0]:
# ==============================================================================
# VALIDACIÓN DE CONTEOS, FILTROS Y MUESTRAS 
# ==============================================================================
from pyspark.sql.functions import col

print("================================================================")
print("EJECUCIÓN DE CONTEOS, FILTROS Y MUESTRAS EN PYSPARK")
print("================================================================")

# 1. Conteo total de registros
total_filas = df_final.count()
print(f"-> 1. Conteo Total usando count(): {total_filas} filas en el dataset.")

# 2. Aplicación de filtro por campo (Canciones con Popularidad mayor o igual a 80)
df_filtrado = df_final.filter(col("Popularity") >= 80)
total_filtrado = df_filtrado.count()
print(f"-> 2. Conteo tras Filtro por Campo (Popularity >= 80): {total_filtrado} canciones encontradas.")

# 3. Muestra de datos aplicando un LIMIT (Mostramos las 3 primeras del filtro)
print("\n-> 3. Muestra de Registros Limitada (LIMIT 3 del subset filtrado):")
df_filtrado.select("Track_Name", "Artist_Name", "Popularity").show(3, truncate=False)

print("----------------------------------------------------------------")
print("PROPÓSITO Y EXPLICACIÓN DE CONTEOS Y MUESTRAS:")
print("-> Propósito: Validar el comportamiento de los operadores de filtro")
print("   y truncado de registros sobre los tipos numéricos del esquema.")
print("-> Resultado: El conteo inicial de 50 filas reafirma que la ingesta")
print("   fue íntegra. El filtro redujo el universo a las canciones de alta")
print("   popularidad y el limit recortó la muestra visual de forma exitosa,")
print("   comprobando que las reglas lógicas operan correctamente.")
print("----------------------------------------------------------------")

#  Análisis Comparativo de Arquitectura: SQL vs Spark (Instrucción 5)

A continuación, se presenta un balance técnico de las ventajas y desventajas entre el uso de SQL tradicional y Apache Spark (PySpark), evaluando criterios de escalabilidad, facilidad de desarrollo y capacidades avanzadas de procesamiento.

### Tabla Comparativa: SQL frente a Spark (PySpark)

| Criterio / Tecnología | SQL (Motor Tradicional / Relacional) | Spark / PySpark (Motor Distribuido) |
| :--- | :--- | :--- |
| **Ventajas Clave** | * **Facilidad de uso:** Curva de aprendizaje baja; sintaxis universalmente conocida.<br>* **Expresividad declarativa:** El desarrollador define *qué* quiere obtener, no *cómo* computarlo.<br>* **Integración nativa con BI:** Conexión directa y fluida con herramientas como Power BI o Tableau. | * **Escalabilidad masiva:** Arquitectura distribuida que procesa Petabytes de datos en clústeres.<br>* **APIs enriquecidas:** Acceso a abstracciones potentes como DataFrames, Datasets y RDDs.<br>* **Ecosistema avanzado:** Soporte nativo para Funciones de Usuario (UDFs) y Machine Learning (MLlib). |
| **Desventajas y Limitaciones** | * **Pipelines complejos:** Se vuelve difícil de mantener o modularizar al encadenar cientos de transformaciones.<br>* **Limitación en UDFs:** Las funciones personalizadas suelen ser lentas o limitadas por el motor de base de datos.<br>* **Falta de flexibilidad:** No es óptimo para procesar datos no estructurados (imágenes, audio, texto libre). | * **Curva de aprendizaje:** Requiere entender conceptos de computación distribuida (shuffling, particionado).<br>* **Ajustes de rendimiento:** Demanda configuración manual de memoria, núcleos y optimización de uniones (joins).<br>* **Latencia en datos pequeños:** Para datasets mínimos, la sobrecarga de coordinar el clúster lo hace más lento que SQL local. |

### Conclusión de Arquitectura
En entornos modernos de Big Data como Databricks, ambas tecnologías no se excluyen, sino que se complementan mediante el optimizador **Catalyst**. SQL destaca por su velocidad para consultas analíticas rápidas y reportería corporativa, mientras que PySpark es la herramienta idónea para construir flujos de ingeniería de datos (ETL) robustos, modulares y escalables en la nube.